# 2.5 記憶範圍怎麼影響預測？


兩句前文是「紅色物體是」與「藍色物體是」，希望分別接「圓」與「方」。如果只把最後一字「是」交給模型，兩題輸入完全一樣，答案卻不同。即使訓練很久，它也無法知道這一回究竟是哪種顏色：關鍵資訊從來沒有傳進去。先檢查可見資料，往往比先把模型加大更有用。

上下文（context）指預測當前字時能夠使用的前文，窗口（window）則是我們實際取出的有限範圍。[2.2](https://birdhackor.github.io/tiny-perceptron-vlm/2.2.html) 把窗口內的字拼成固定長卡，所以窗口越長，可能保留更遠的線索；但眼前要知道多長才夠，而不是假定越長越好。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [("紅色物體是", "圓"), ("藍色物體是", "方")]
for length in [1, 3, 5]:
    visible = []
    for prefix, answer in examples:
        visible.append((prefix[-length:], answer))
    print(length, visible)

輸入是兩組「前文、答案」，用Python的兩項小組（tuple）保存；圓括號裡兩項的次序固定。每輪 `length` 指模型可見最近幾字。`prefix[-length:]` 從倒數第length個位置取到最後，這是 [1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html) 切片的延伸，不是反過來讀文字。`visible` 逐筆保存目前能見到的片段與應答。

輸出第一輪是 `('是','圓')` 與 `('是','方')`；第三字窗口則都是 `物體是`，仍不能區分。五字窗口纔看到完整的 `紅色物體是` 與 `藍色物體是`。模型給相同輸入就只能算出相同的一套候選分配；它可以同時給圓與方各一些機率，卻不能在沒有顏色線索時穩定挑對每題的唯一答案。隨機選擇也不會憑空補回被刪掉的資訊。

這叫可辨識性問題：資料中是否有足以區分答案的線索。對這兩題，四字窗口其實也不夠，因為 `色物體是` 仍相同；必須到五字。反過來，即使五字已能辨識，也只是「有機會學」，不是「一定已學會」。模型如何用顏色建立規則、訓練資料是否夠、是否記住了特例，仍要另外驗證。

擴大固定窗口還會增加計算與參數。若每字D個特徵、窗口C字，拼接後有C×D個輸入；接H個輸出的Linear需要H×C×D個權重。這裡只需讀成「每個輸出都為每個輸入保存一個權重」；例如D=2、C從3變6，其他固定，輸入層權重也翻倍。更長窗口有代價，要用任務中線索的距離判斷是否值得。

練習把兩句前文改成「紅色的小小物體是」與「藍色的小小物體是」，保持答案不變。先手數顏色區別字離末尾多遠，再加入窗口長度8、7進行核對；區別紅與藍的是第一個字，整句八字，所以八字能區分、七字仍不能。你不是在訓練模型，而是在確認題目有沒有把所需線索交出去。
